# S10

## Parâmetros FrodoKEM-640

In [2]:
import hashlib
import random

n = 640
q = 2**15 
n_linhas_colunas = 8
B = 2
T_CHI640 = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]

Zq = IntegerModRing(q)

## Tarefa 1 — `genMat`

In [3]:
def genMat(seedA, n, Zq):
    seed_int = int.from_bytes(seedA, byteorder = 'big')
    set_random_seed(seed_int)
    return random_matrix(Zq, n, n)

data = b"ola mundo"

shake = hashlib.shake_256(data)
seedA = shake.digest(int(16))

A = genMat(seedA, n, Zq)
print(A[:5, :5])


[27408   175  5180 30966  5514]
[27328 11834  8848 21163 31501]
[ 6715 15888 23901   491 15829]
[26260 23766 31006 14482 18240]
[ 3251  3538  8810  2566 27767]


## Tarefa 2 — `chiSample`

In [4]:
def chiSample(T, nlinhas, ncolunas, Zq, q, k):
    matriz = []
    for i in range(nlinhas):
        linha = []
        for j in range(ncolunas):
            x = randint(0, 2**k - 1)
            
            valor_tabela = len(T) - 1
            for idx in range(len(T)):
                if x <= T[idx]:
                    valor_tabela = idx
                    break
            
            s = randint(0, 1)
            valor = (1 - 2*s) * valor_tabela
            linha.append(valor)
    
        matriz.append(linha)
    
    return matrix(Zq, nlinhas, ncolunas, [v for linha in matriz for v in linha])

chiSample(T_CHI640, 5, 5, Zq, q, 15)

[32765     0 32767     1     1]
[32760 32762 32767     2     0]
[32762     1     5     2 32764]
[32767 32766     1     0     2]
[    0     0     1     2 32763]

## Tarefa 3 — `encode` / `decode`

In [5]:
def Bits_para_numero(lista_bits):
    i = len(lista_bits) - 1
    elevado = 0
    numero = 0
    
    while i >= 0:
        numero += lista_bits[i] * 2**elevado
        elevado += 1
        i -= 1

    return numero

def encode(msg_bits, n_linhas_colunas, B, q, Zq):
    entrada = []

    i = 0
    while i < len(msg_bits):
        sub_msg_bits = msg_bits[i: i + B]
        
        x = Bits_para_numero(sub_msg_bits)

        entrada.append(Zq(x * (q // 2**B)))
        
        i += B

    return matrix(Zq, n_linhas_colunas, n_linhas_colunas, entrada)

def Convertar_para_binario(numero, B):
    bits = [0] * B      
    valor = numero
    for i in range(B - 1, -1, -1):
        bit = valor % 2       
        valor = valor // 2    
        bits[i] = bit
    return bits

def decode(M, n_linhas_colunas, B, q):
    res = []
    for i in range(n_linhas_colunas):
        for j in range(n_linhas_colunas):
            valor = round((int(M[i, j]) * (2**B)) / q)
            res += Convertar_para_binario(valor, B)
    return res

msg_bits  = [randint(0, 1) for i in range(128)]
M_enc     = encode(msg_bits, n_linhas_colunas, B, q, Zq)
msg_final  = decode(M_enc, n_linhas_colunas, B, q)

if msg_bits == msg_final: 
    print(True)
else:
    print(False)

True


## Tarefa 4 — PKE (KeyGen, Enc, Dec)

In [6]:
def keyGen(n = n, n_linhas_colunas = n_linhas_colunas, q = q, T = T_CHI640, Zq = Zq):
    seedA = bytes([randint(0, 255) for _ in range(16)])
    A = genMat(seedA, n, Zq)
    S = chiSample(T, n, n_linhas_colunas, Zq, q, 15)
    E = chiSample(T, n, n_linhas_colunas, Zq, q, 15)
    B_mat = A * S + E
    pk = (seedA, B_mat)
    sk = S
    return sk, pk

def enc(pk, msg_bits, n = n, n_linhas_colunas = n_linhas_colunas, q = q, B_bits = B, T = T_CHI640, Zq = Zq):
    seedA, B_mat = pk
    A = genMat(seedA, n, Zq)                                     
    Sp = chiSample(T, n_linhas_colunas, n, Zq, q, 15)                
    Ep = chiSample(T, n_linhas_colunas, n, Zq, q, 15)                  
    Epp = chiSample(T, n_linhas_colunas, n_linhas_colunas, Zq, q, 15)  
    C1 = Sp * A + Ep
    Vp = Sp * B_mat + Epp
    M  = encode(msg_bits, n_linhas_colunas, B_bits, q, Zq) 
    C2 = Vp + M
    return C1, C2

def dec(sk, C, n_linhas_colunas = n_linhas_colunas, q = q, B_bits = B, Zq = Zq):
    C1, C2 = C
    V = C1 * sk
    M = C2 - V
    return decode(M, n_linhas_colunas, B_bits, q) 

In [7]:
msg = [randint(0, 1) for _ in range(128)]
sk, pk = keyGen()

C = enc(pk, msg)
msg_dec = dec(sk, C)

print(f"Mensagem original (primeiros 16 bits): {msg[:16]} (128 bits)")
print(f"Mensagem decifrada (primeiros 16 bits): {msg_dec[:16]}")
print(f"Correto? : {msg == msg_dec}")

Mensagem original (primeiros 16 bits): [0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0] (128 bits)
Mensagem decifrada (primeiros 16 bits): [0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0]
Correto? : True


## Parâmetros miniFrodoKEM

In [8]:
n_mini = 64
q_mini = 2**10
n_bar_mini = 8
B_mini = 2
T_CHI64 = [6536, 14465, 16234, 16379, 16384]

Zq_mini = IntegerModRing(q_mini)

## Tarefa 5 — miniPKE

In [9]:
def miniKeyGen(n = n_mini, n_bar_mini = n_bar_mini, q = q_mini, T = T_CHI64, Zq = Zq_mini):
    seedA = bytes([randint(0, 255) for _ in range(16)])
    A = genMat(seedA, n, Zq)
    S = chiSample(T, n, n_bar_mini, Zq, q, 14)
    E = chiSample(T, n, n_bar_mini, Zq, q, 14)
    B_mat = A * S + E
    pk = (seedA, B_mat)
    sk = S
    return sk, pk
   
def miniEnc(pk, msg_bits, n = n_mini, n_bar_mini = n_bar_mini, q = q_mini, B_bits = B_mini, T = T_CHI64, Zq = Zq_mini):
    seedA, B_mat = pk
    A = genMat(seedA, n, Zq)                                     
    Sp = chiSample(T, n_bar_mini, n, Zq, q, 14)                
    Ep = chiSample(T, n_bar_mini, n, Zq, q, 14)                  
    Epp = chiSample(T, n_bar_mini, n_bar_mini, Zq, q, 14)  
    C1 = Sp * A + Ep
    Vp = Sp * B_mat + Epp
    M  = encode(msg_bits, n_bar_mini, B_bits, q, Zq) 
    C2 = Vp + M
    return C1, C2

def miniDec(sk, C, n_bar_mini = n_bar_mini, q = q_mini, B_bits = B_mini):
    C1, C2 = C
    V = C1 * sk
    M = C2 - V
    return decode(M, n_bar_mini, B_bits, q)

In [10]:
msg2         = [randint(0, 1) for _ in range(128)]
sk2, pk2     = miniKeyGen()
C2           = miniEnc(pk2, msg2)
msg2_dec     = miniDec(sk2, C2)

print(f"Mensagem original : {msg2[:16]}...")
print(f"Mensagem decifrada: {msg2_dec[:16]}...")
print(f"Correto? : {msg2 == msg2_dec}")

Mensagem original : [0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0]...
Mensagem decifrada: [0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0]...
Correto? : True


## Tarefa 6 — Impacto do ruído

In [11]:
def miniKeyGen_ruido(n = n_mini, n_bar_mini = n_bar_mini, q = q_mini, T = T_CHI640, Zq = Zq_mini):
    seedA = bytes([randint(0, 255) for _ in range(16)])
    A = genMat(seedA, n, Zq)
    S = chiSample(T, n, n_bar_mini, Zq, q, 15) 
    E = chiSample(T, n, n_bar_mini, Zq, q, 15)
    B_mat = A * S + E
    pk = (seedA, B_mat)
    sk = S
    return sk, pk
   
def miniEnc_ruido(pk, msg_bits, n = n_mini, n_bar_mini = n_bar_mini, q = q_mini, B_bits = B_mini, T = T_CHI640, Zq = Zq_mini):
    seedA, B_mat = pk
    A = genMat(seedA, n, Zq)                                     
    Sp = chiSample(T, n_bar_mini, n, Zq, q, 15)                
    Ep = chiSample(T, n_bar_mini, n, Zq, q, 15)                  
    Epp = chiSample(T, n_bar_mini, n_bar_mini, Zq, q, 15)  
    C1 = Sp * A + Ep
    Vp = Sp * B_mat + Epp
    M  = encode(msg_bits, n_bar_mini, B_bits, q, Zq) 
    C2 = Vp + M
    return C1, C2

In [12]:
N_tests = 100
errors  = 0

for _ in range(N_tests):
    msg_t      = [randint(0, 1) for _ in range(128)]
    sk_t, pk_t = miniKeyGen_ruido()
    Ct         = miniEnc_ruido(pk_t, msg_t)
    dec_t      = miniDec(sk_t, Ct)
    if msg_t != dec_t:
        errors += 1

print(f"Testes realizados : {N_tests}")
print(f"Erros de decifra : {errors}  ({100 * errors / N_tests}%)")

Testes realizados : 100
Erros de decifra : 100  (100%)


### Comentário — Impacto do ruído

| Esquema | n | q | σ (ruído) | Taxa de erro |
|---|---|---|---|---|
| **FrodoKEM-640** | 640 | 2¹⁵ = 32768 | ≈ 2.8 | ≈ 0% |
| **miniPKE** (T_CHI64) | 64 | 2¹⁰ = 1024 | ≈ 1.0 | ≈ 0% |
| **miniPKE** (T_CHI640) | 64 | 2¹⁰ = 1024 | ≈ 2.8 | **≈ 100%** |

**Conclusão:** Com `q = 2¹⁰` e ruído `σ ≈ 2.8`, o ruído acumulado nas multiplicações matriciais (`n=64` somas) ultrapassa o limiar de descodificação (`q / 2^(B+1) = 128`). No FrodoKEM-640 original, o `q` muito maior (32768) cria uma "margem" suficiente para absorver o ruído e manter a taxa de erro negligenciável.

**Nota:** Durante a realização deste trabalho, recorremos ao assistente de inteligência artificial Claude para auxiliar na compreensão de conceitos. O comentário acima resulta de uma síntese gerada por essa ferramenta, que decidimos incluir pela sua clareza e pertinência para a análise.